In [ ]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia_parsed = pd.read_csv(
    "../data/processed/fixed/srodmiescie_short.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

data_gdynia_unparsed = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv", sep=";", index_col=0, dtype=column_types
)

1.

In [ ]:
print(f"unparsed type: {data_gdynia_unparsed['date_of_checking'].dtype}")
print(f"parsed type: {data_gdynia_parsed['date_of_checking'].dtype}")

print(f"min_unparsed: {data_gdynia_unparsed['date_of_checking'].min()}")
print(f"max_unparsed: {data_gdynia_unparsed['date_of_checking'].max()}")
print(f"min_parsed: {data_gdynia_parsed['date_of_checking'].min()}")
print(f"max_parsed: {data_gdynia_parsed['date_of_checking'].max()}")

In [ ]:
data_gdynia_parsed["date_of_checking"].dt.day

In [ ]:
pd.testing.assert_series_equal(
    data_gdynia_parsed["date_of_checking"],
    pd.to_datetime(data_gdynia_unparsed["date_of_checking"]),
)
assert data_gdynia_parsed["date_of_checking"].nunique() == 7
assert data_gdynia_parsed["date_of_checking"].isna().sum() == 0

2.

In [ ]:
data_gdynia_parsed["date_of_checking_daynumber"] = data_gdynia_parsed[
    "date_of_checking"
].dt.day_of_week
data_gdynia_parsed["date_of_checking_dayofweek"] = data_gdynia_parsed[
    "date_of_checking"
].dt.day_name()
data_gdynia_parsed

In [ ]:
data_gdynia_parsed.groupby(
    ["date_of_checking_dayofweek", "date_of_checking_daynumber"]
).size()

In [ ]:
(data_gdynia_parsed["date_of_checking_daynumber"] >= 5).mean()

In [ ]:
pd.Timestamp("2026-09-23").day_name()

In [ ]:
assert data_gdynia_parsed.groupby("date_of_checking_dayofweek").size().sum() == 982
assert np.isclose(
    len(data_gdynia_parsed.loc[data_gdynia_parsed["date_of_checking_daynumber"] >= 5])
    / data_gdynia_parsed.groupby("date_of_checking_dayofweek").size().sum(),
    0.571,
    atol=0.01,
)

3.

In [ ]:
data_gdynia_parsed["week_of_checking"] = data_gdynia_parsed[
    "date_of_checking"
].dt.isocalendar()["week"]
buil_per_week = data_gdynia_parsed.groupby("week_of_checking").size()

In [ ]:
assert buil_per_week.sum() == 982
assert buil_per_week.loc[39] == 788
assert buil_per_week.loc[40] == 194

4.

In [ ]:
REF = pd.Timestamp("2026-10-17")

In [ ]:
data_gdynia_parsed["days_since_checking"] = (
    REF - data_gdynia_parsed["date_of_checking"]
).dt.days
data_gdynia_parsed["days_since_checking"].describe()

In [ ]:
assert data_gdynia_parsed["days_since_checking"].min() == 17
assert data_gdynia_parsed["days_since_checking"].max() == 24
assert np.isclose(data_gdynia_parsed["days_since_checking"].mean(), 20.21, atol=0.01)
assert (data_gdynia_parsed["days_since_checking"] > 20).sum() == 327

5.

In [ ]:
data_groupby = data_gdynia_parsed.groupby("date_of_checking").size()

data_resample = data_gdynia_parsed.set_index("date_of_checking").resample("D").size()
data_resample

In [ ]:
assert data_resample.loc["2026-09-28"] == 0
assert data_resample.sum() == 982
pd.testing.assert_series_equal(data_groupby, data_resample.loc[data_resample != 0])

6.

In [ ]:
date_resample_6 = (
    data_gdynia_parsed.assign(conf_3=lambda d: d["confidence"] == 3)
    .resample("D", on="date_of_checking")
    .agg(
        num_build=("osm_id", "size"),
        num_streets=("addr_street", "nunique"),
        share_build_conf_3=("conf_3", "mean"),
    )
)

In [ ]:
date_resample_6.loc["2026-09-27", "num_build"] == 461
date_resample_6.loc["2026-09-27", "num_streets"] == 34


In [ ]:
assert date_resample_6.sort_values("share_build_conf_3", ascending=False).index[
    0
] == pd.Timestamp("2026-09-25")

7.

In [ ]:
data_date_style = (
    data_gdynia_parsed.groupby(
        [pd.Grouper(key="date_of_checking", freq="D"), "arch_style"]
    )
    .size()
    .unstack(fill_value=0)
)
data_date_style

In [ ]:
data_date_style.sum(axis=1)

In [ ]:
assert (data_groupby == (data_date_style.sum(axis=1))).all()
assert data_date_style.loc["2026-09-27", "modernizm"] == 222
assert data_date_style.loc["2026-09-27", "współczesność"] == 228

8.

In [ ]:
data_weekend = (
    data_gdynia_parsed.set_index("date_of_checking")
    .sort_index()
    .loc["2026-09-26":"2026-09-27"]
)

In [ ]:
data_weekend

this slicing differ from [a:b] method becouse right part in dates or other labels is inclusive where normally its not inclusive

In [ ]:
assert len(
    data_gdynia_parsed.loc[
        data_gdynia_parsed["date_of_checking"].between("2026-09-26", "2026-09-27")
    ]
) == len(data_weekend)

9.

In [ ]:
data_gdynia_dates = data_gdynia_parsed.groupby("addr_street").agg(
    first_date=("date_of_checking", "min"),
    last_date=("date_of_checking", "max"),
    num_dates=("date_of_checking", "nunique"),
).assign(
    range_dates=lambda d: (d["last_date"] - d["first_date"]).dt.days,
    days_since_last=lambda d: (REF - d["last_date"]).dt.days,
)
data_gdynia_dates.loc[data_gdynia_dates['days_since_last']>20]

In [ ]:
assert (data_gdynia_dates.loc['Świętojańska'] == [pd.Timestamp("2026-09-24"),pd.Timestamp("2026-09-30"),5,6,17]).all()
assert len(data_gdynia_dates) == 44
assert len(data_gdynia_dates.loc[data_gdynia_dates['days_since_last']>20]) == 5

10.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")

ax.bar(data_resample.index.strftime('%a,%d.%m'), data_resample.values)
ax.set_xlabel('days of checking')
ax.set_ylabel('number of checked buildings')
ax.set_title('Number of buildings checked per day')
plt.show()